### IMPORTING LIBRARY

In [1]:
import os
import random
import datetime
import json
import h5py
import umap
import pandas                   as pd
import numpy                    as np
import plotly.express           as px
import plotly.graph_objects     as go

import matplotlib.pyplot        as plt

from tqdm                       import tqdm
from preProcessingData          import PreProcessData
from utility.matpilotDraw       import MatPlot
from utility.plotloyDraw        import PlotlyDraw
from prettytable                import PrettyTable
from model.vaeConv1DCustom      import vaeConv1DCustom 
from vaeTrainer                 import appVAE
from mpl_toolkits.mplot3d       import Axes3D
from sklearn.cluster            import KMeans
from sklearn.cluster            import DBSCAN


matPlot    = MatPlot() 
plotly     = PlotlyDraw()

/home/ducan/anaconda3/envs/tensorflowStable/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-07-21 20:51:15.426496: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-07-21 20:51:15.614656: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-07-21 20:51:16.274947: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot ope

In [2]:
preData    = PreProcessData()

### DATA LOADING

In [ ]:
# Use this one only aiming to show the data summary 
rawData = preData.dataLoadingCriticalData()

In [ ]:
# Use this one only aiming to save to data into .h5 files
preData.save2H5Format(rawData)

In [3]:
rawDataH5 = preData.loadDataFromH5Format()
print(rawDataH5.info())

Process:270714||2024-07-21 20:51:17,641-INFO [preProcessingData.py:loadDataFromH5Format():224]:Loading data from H5 files ...
Process:270714||2024-07-21 20:51:19,529-INFO [preProcessingData.py:loadDataFromH5Format():227]:Finish loading data from H5 files ...


<class 'pandas.core.frame.DataFrame'>
Index: 321435 entries, 0 to 321434
Data columns (total 41 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   tiempo                         321435 non-null  datetime64[ns]
 1   Descripción Equipo             30756 non-null   object        
 2   Modo de Falla                  30756 non-null   object        
 3   Comentario                     11837 non-null   object        
 4   Especialidad                   30756 non-null   object        
 5   Motivo                         30756 non-null   object        
 6   Inicio Reportado               30756 non-null   datetime64[ns]
 7   Fin Reportado                  30756 non-null   datetime64[ns]
 8   corriente(A)                   321435 non-null  float64       
 9   velocidad(rpm)                 321435 non-null  float64       
 10  nivel_tolva_compensacion(%)    321435 non-null  float64       
 11  Poten

### DATA CLEANING

#### Manipulated and efficiency Variables

In [4]:
preData.splitInputandEfficiencyVar()

+-------------------------------+-----------------------------+
|         ManupulatedVar        |        EfficiencyVar        |
+-------------------------------+-----------------------------+
|          corriente(A)         | nivel_tolva_compensacion(%) |
|         velocidad(rpm)        |       rendimiento(ton)      |
|          Potencia(kW)         |        Piñón_ext(°C)        |
|       posicion_poste(%)       |        Piñón_int(°C)        |
|   presion_estanque_aire(bar)  |       Piñón_int(mm/s)       |
|      Eje_piñón_aceite(%)      |       Piñón_ext(mm/s)       |
|   Lubricación_principal(psi)  |   Lubricación_retorno(°C)   |
|    Lubricación_principal(A)   |   Lubricación_depósito(°C)  |
|   Lubricación_respaldo(psi)   |   Lubricación_entrada(°C)   |
|    Lubricación_respaldo(A)    |                             |
| Lubricación_post_filtros(psi) |                             |
|  Lubricación_válvula_MPS(psi) |                             |
|  Lubricación_entrada_sup(psi) |       

##### Draw variables

In [ ]:
# ============================
# Manipulated variables
# ----------------------------
for idxFeature in preData.manipulatedCr1:
    drawedData = preData.crusher1[idxFeature].to_numpy().astype(float)
    matPlot.draw1D(yCoor=drawedData,
                   title=idxFeature)

In [ ]:
# ============================
# Efficiency variables
# ----------------------------
for idxFeature in preData.efficiencyCr1:
    drawedData = preData.crusher1[idxFeature].to_numpy().astype(float)
    matPlot.draw1D(yCoor=drawedData,
                   title=idxFeature)

#### Sumarry MOF

In [ ]:
preData.sumarryMOF()

#### Data summarization

In [ ]:
markedPoints = preData.sumarryDataFromFile()

#### Add efficiency variables

In [7]:
addedVariables = preData.additionEfficiencyVar(rawDataH5)

print(addedVariables.info())
print(addedVariables.isna())

29
<class 'pandas.core.frame.DataFrame'>
Index: 321435 entries, 0 to 321434
Data columns (total 49 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   tiempo                         321435 non-null  datetime64[ns]
 1   Descripción Equipo             30756 non-null   object        
 2   Modo de Falla                  30756 non-null   object        
 3   Comentario                     11837 non-null   object        
 4   Especialidad                   30756 non-null   object        
 5   Motivo                         30756 non-null   object        
 6   Inicio Reportado               30756 non-null   datetime64[ns]
 7   Fin Reportado                  30756 non-null   datetime64[ns]
 8   corriente(A)                   321435 non-null  float64       
 9   velocidad(rpm)                 321435 non-null  float64       
 10  nivel_tolva_compensacion(%)    321435 non-null  float64       
 11  Po

## BUIDING MODEL

In [ ]:
norData = preData.normalizeData(rawDataH5[preData.crusher1Value])    # Normalize the data

In [ ]:
# ============================
# Manipulated variables
# ----------------------------
for idxFeature in preData.manipulatedCr1:
    drawedData = norData[idxFeature].to_numpy().astype(float)
    matPlot.draw1D(yCoor=drawedData,
                   title=idxFeature)

In [ ]:
# ============================
# Efficiency variables
# ----------------------------
for idxFeature in preData.efficiencyCr1:
    drawedData = norData[idxFeature].to_numpy().astype(float)
    matPlot.draw1D(yCoor=drawedData,
                   title=idxFeature)

### Training


In [ ]:
healthyStateData,healthyPos        = preData.getHealthyState(norData)

In [ ]:
# ============================
# Manipulated variables
# ----------------------------
for idxFeature in preData.manipulatedCr1:
    drawedData = healthyStateData[idxFeature].to_numpy().astype(float)
    matPlot.draw1D(yCoor=drawedData,
                   title=idxFeature)

#### Model

In [ ]:
with open('model/modelConfig.json', 'r') as configFile:
    config = json.load(configFile)

vaeConfig = config["vae"]
 
# inputX                  = healthyStateData[preData.manipulatedCr1]
# outputY                 = healthyStateData[preData.efficiencyCr1]

# trainXY,valXY           = preData.splitTrainVal(inputX,outputY)                     # Split input and efficiency variables into train and validation dataset

# seqTrainXY              = preData.splitAndPadSequences(trainXY)                     # Split them into sequences
# seqValXY                = preData.splitAndPadSequences(valXY)                       # Split them into sequences

inputX                  = healthyStateData[preData.manipulatedCr1]
trainXY,valXY           = preData.splitTrainVal(inputX  = inputX,
                                                outputY = inputX)                   # Split input and efficiency variables into train and validation dataset
seqTrainXY              = preData.splitAndPadSequences(trainXY)                     # Split them into sequences
seqValXY                = preData.splitAndPadSequences(valXY)                       # Split them into sequences

model                   = vaeConv1DCustom(vaeConfig)

print("The size of train data:  ",seqTrainXY[0].shape ,"(input)    ", seqTrainXY[1].shape,"(output)")
print("The size of val data:    ",seqValXY[0].shape ,"(input)    ", seqValXY[1].shape,"(output)")

model.summary()

### GET RECONSTRUCTION ERROR AND LATENT SPACE

In [ ]:
# Predict to get the reconstruction error and latent space result
inputX                  = norData[preData.manipulatedCr1]
outputY                 = norData[preData.efficiencyCr1]
seqData                 = preData.splitAndPadSequences([inputX,inputX])                       # Split them into sequences

lossResults,latentResults   = appVAE.predict(seqData,vaeConfig)

print(len(lossResults))

In [ ]:
npLossResults           = np.array(lossResults)
npLatentResults         = np.array(latentResults)
with h5py.File('ckpt/otherResults/VAECNN1D_ManupulatedAndEfficiencyInputOutput[200, 100, 50]_[50, 100, 200]latent_2_lossAndLatent.h5', 'w') as file:
    # Create datasets in the file
    file.create_dataset('lossResults',      data = npLossResults)
    file.create_dataset('latentResults',    data = npLatentResults)

#### Check results

In [ ]:
drawedData = preData.crusher1["corriente(A)"].to_numpy().astype(float)
time       = preData.crusher1["tiempo"]
# print(time)
# print(len(time))
# print(len(drawedData))

with h5py.File('ckpt/otherResults/VAECNN1D_ManupulatedAndEfficiencyInputOutput[200, 100, 50]_[50, 100, 200]latent_2_lossAndLatent.h5', 'r') as file:
    npLossResults = file['lossResults'][:]
    npLatentResults = file['latentResults'][:]

padSize             = len(drawedData) - len(npLossResults)

npLossResultsPad    = np.pad(npLossResults, (padSize, 0), mode='constant', constant_values=0)
npLatentResultsPad  = np.pad(npLatentResults, (padSize, 0), mode='constant', constant_values=0)

plotly.plot_matrix(xData                = time,
                   yData                = [drawedData,npLossResultsPad,npLatentResultsPad],
                   title                = ["corriente(A)","Loss","Latent space"],
                   highLighHtPos        = healthyPos,
                   makeImportantEvent   = markedPoints,
                   withShap             = True
                   ) 

# addedVariables["constructionLoss"] = npLossResultsPad

In [ ]:
matPlot.draw2D( xCoor                = time[158700:164472],
                yCoor                = npLossResultsPad[158700:164472],
                title                = ["Loss"],      
                ) 
matPlot.draw2D( xCoor                = time[198160:198519],
                yCoor                = npLossResultsPad[198160:198519],
                title                = ["Loss"],      
                ) 

matPlot.draw2D( xCoor                = time[284798:293894],
                yCoor                = npLossResultsPad[284798:293894],
                title                = ["Loss"],      
                ) 

matPlot.draw2D( xCoor                = time[168858,169727],
                yCoor                = npLossResultsPad[168858,169727],
                title                = ["Loss"],      
                ) 



### D. UMAP for entire dataset

In [ ]:
manipulatedCr1  = preData.crusher1[preData.manipulatedCr1]
maWTimeCr1      = manipulatedCr1[:]
print("The input variables shape",maWTimeCr1.shape)
umapReduction   = umap.UMAP(    n_components    = 3,
                                n_neighbors     = 15,
                                min_dist        = 0.1)
maEmbedding     = umapReduction.fit_transform(maWTimeCr1)

In [ ]:
kmeans = KMeans(n_clusters=4)
kmeans.fit(maEmbedding)

labels = kmeans.labels_
centers = kmeans.cluster_centers_

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# Color map for clusters
colors = ['r', 'g', 'b', 'y']

# Plot each data point
for i in range(len(maEmbedding)):
    ax.scatter(maEmbedding[i, 0], maEmbedding[i, 1], maEmbedding[i, 2], c=colors[labels[i]], marker='o')

# Plot the centroids
for i in range(4):
    ax.scatter(centers[i, 0], centers[i, 1], centers[i, 2], c=colors[i], marker='x', s=100, linewidths=3)

ax.set_title('Mode of condition')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.set_zlabel('Feature 3')
plt.show()

In [ ]:
efficiencyLabel = preData.efficiencyCr1 + preData.addEfficiencyCr1 +  ["constructionLoss"]

efficiencyCr1   = addedVariables[efficiencyLabel]
effWTimeCr1     = efficiencyCr1[:]
print("The efficieny variables shape",effWTimeCr1.shape)

umapReduction1 = umap.UMAP( n_components=3,
                            n_neighbors=15,
                            min_dist=0.1)

effEmbedding      = umapReduction1.fit_transform(effWTimeCr1)

In [ ]:
kmeans1 = KMeans(n_clusters=4)
kmeans1.fit(effEmbedding)

labels = kmeans1.labels_
centers = kmeans1.cluster_centers_

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# Color map for clusters
colors = ['r', 'g', 'b', 'y']

# Plot each data point
for i in range(len(effEmbedding)):
    ax.scatter(effEmbedding[i, 0], effEmbedding[i, 1], effEmbedding[i, 2], c=colors[labels[i]], marker='o')

# Plot the centroids
for i in range(4):
    ax.scatter(centers[i, 0], centers[i, 1], centers[i, 2], c=colors[i], marker='x', s=100, linewidths=3)

ax.set_title('Mode of condition')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.set_zlabel('Feature 3')
plt.show()

In [ ]:
dbscan = DBSCAN(eps=2.6, min_samples=60)
clusters = dbscan.fit_predict(effEmbedding)
print("Cluster labels:", clusters)


# Prepare the 3D plot
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# Get unique labels and number of colors needed
unique_labels = np.unique(clusters)

# Generate colors for each cluster
colors = plt.cm.jet(np.linspace(0, 1, len(unique_labels)))

# Plot each cluster with a unique color
for k, col in zip(unique_labels, colors):
    if k == -1:
        # Black used for noise.
        col = 'k'

    class_member_mask = (clusters == k)

    # Plot the datapoints that are clustered
    xyz = effEmbedding[class_member_mask]
    ax.scatter(xyz[:, 0], xyz[:, 1], xyz[:, 2], c=[col], edgecolor='k', s=50, label=f'Cluster {k}')

# Plot title and labels
ax.set_title('DBSCAN Clustering')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')

# Legend
ax.legend()

# Show plot
plt.show()



#### c.1 Draw UMAP

In [ ]:
# matPlot.draw3D(yCoor1 = maEmbedding[:,0],
#                yCoor2 = maEmbedding[:,1],
#                yCoor3 = maEmbedding[:,2],
#                title  = "UMAP for Manipulated variables")

matPlot.draw3D(yCoor1 = effEmbedding[:,0],
               yCoor2 = effEmbedding[:,1],
               yCoor3 = effEmbedding[:,2],
               title  = "UMAP for efficient variables")

In [ ]:
# maEmbedding_df  = pd.DataFrame(maEmbedding, columns=['UMAP_1', 'UMAP_2', 'UMAP_3'])
effEmbedding_df = pd.DataFrame(effEmbedding, columns=['UMAP_1', 'UMAP_2', 'UMAP_3'])
# print(maEmbedding_df)
# print(effEmbedding_df)
# fig = px.scatter_3d(
#     maEmbedding_df,
#     x='UMAP_1',
#     y='UMAP_2',
#     z='UMAP_3',
#     title='3D UMAP Embedding with Plotly'
# )

# fig.update_traces(marker=dict(size=3))
# fig.show()

fig = px.scatter_3d(
    effEmbedding_df,
    x='UMAP_1',
    y='UMAP_2',
    z='UMAP_3',
    title='3D UMAP Embedding with Plotly'
)

fig.update_traces(marker=dict(size=3))
fig.show()